## Fisher estimates for review test cases

Goal: estimate Fisher information contained in the review test cases, to compare against PE run estimates.

Problem: installing the `stableemrifisher` package inside the PE search environment probably breaks this environment, which is not ideal as it was a pain to set up. First test notebook here with the pre-compiled binaries of the packages instead of the source-builds of specific commit messages.

### Environment

Run this with the **`few-1PAT1R`** kernel. Relevant versions there:

| package | version |
|---|---|
| `fastemriwaveforms` | dev install from `FEW/1PAT1R/FEW-dev` |
| `fastlisaresponse` | 1.1.17 (pre-compiled wheel) |
| `lisaanalysistools` | 1.2.8 (pre-compiled wheel) |
| `stableemrifisher` | editable from `Projects/StableEMRIFisher` |

`fastlisaresponse` 1.1.17 is **not** the API used by `validation/PE_test_runs/src/waveform_updated.py`.
Differences that matter here:

* `ResponseWrapper` takes `force_backend=("cpu"|"cuda11x"|"cuda12x")`, **not** `use_gpu=`.
  Passing `use_gpu=` lands in `**kwargs` and is forwarded to `pyResponseTDI`, which rejects it.
* `orbits=` must be an *instance* of `lisatools.detector.Orbits` (the signature's default is the
  *class* `EqualArmlengthOrbits`, which would fail its own `isinstance` assert).
* There is no `t_buffer` argument; `t0` is simply the garbage-removal buffer in seconds.
* `__call__` returns a **list** of TDI channels.

Three fixes were needed to get this combination running; all of them are already applied:

1. **Duplicate `libstdc++` abort.** The `fastlisaresponse` and `lisatools` wheels each vendor their
   own copy of `libstdc++.6.dylib` (and `libgcc_s.1.1.dylib`) under `<pkg>/.dylibs/`. Loading both
   C++ backends in one process makes dyld map two copies of the GNU C++ runtime and the process
   dies with `Fatal Python error: Aborted` — in *either* import order. Fixed by pointing
   `fastlisaresponse/.dylibs/*` at the `lisatools` copies:

   ```bash
   SP=$(python -c "import site; print(site.getsitepackages()[0])")
   cd $SP/fastlisaresponse/.dylibs
   ln -sf ../../lisatools/.dylibs/libstdc++.6.dylib  libstdc++.6.dylib
   ln -sf ../../lisatools/.dylibs/libgcc_s.1.1.dylib libgcc_s.1.1.dylib
   ```

   **A `pip install --force-reinstall fastlisaresponse` will undo this and the abort comes back.**

2. `stableemrifisher.noise` imported `lisatools` at module import time, so `import stableemrifisher`
   died in any environment without it. The import is now deferred into `write_psd_file`.

3. `stableemrifisher.fisher` built its default PSD path as `os.getcwd() + PSD_filename` (no
   separator), dropping the file in the *parent* directory under a mangled name. Now `os.path.join`.

The cell below asserts fix 1 is in place before anything else is imported.

In [ ]:
import os
import site

_sp = site.getsitepackages()[0]
_flr_dylib = os.path.join(_sp, "fastlisaresponse", ".dylibs", "libstdc++.6.dylib")
_lt_dylib = os.path.join(_sp, "lisatools", ".dylibs", "libstdc++.6.dylib")
if os.path.exists(_flr_dylib) and os.path.exists(_lt_dylib):
    assert os.path.realpath(_flr_dylib) == os.path.realpath(_lt_dylib), (
        "fastlisaresponse and lisatools ship separate copies of libstdc++; importing both "
        "C++ backends will abort the kernel. See the markdown cell above for the symlink fix."
    )
    print("libstdc++ de-duplicated ->", os.path.realpath(_flr_dylib))

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

from few.utils.constants import YRSID_SI
from few.waveform import Circ1PAT1R, GenerateEMRIWaveform
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits, ESAOrbits

from stableemrifisher.fisher import StableEMRIFisher

import fastlisaresponse
import lisatools

print("fastlisaresponse", fastlisaresponse.__version__)
print("lisatools       ", lisatools.__version__)

USE_GPU = False                                  # no CuPy on this machine
FORCE_BACKEND = "cuda12x" if USE_GPU else "cpu"

CONFIG_DIR = Path("../config/review").resolve()
OUT_DIR = Path("fisher_out").resolve()
OUT_DIR.mkdir(exist_ok=True)
print("configs:", CONFIG_DIR)

### Review test-case parameters

Copied from `validation/PE_test_runs/config/review/Review_test_{1,2,3}_inj_1PA_rec_1PA.yaml`.

Two things are *not* verbatim copies of the YAML and are worth being explicit about:

* **Masses are redshifted.** `PE_response_updated.py::_emri_vector` multiplies `M` and `mu` by
  `(1 + z)` before handing them to FEW, so the YAML holds source-frame masses and the waveform sees
  detector-frame ones. The dicts below store the **detector-frame** values, i.e. what the sampler
  actually conditions on. `d_L` is passed through unchanged (Gpc).
* **Names are translated** to the `stableemrifisher` convention
  (`M -> m1`, `mu -> m2`, `d_L -> dist`, `theta_S -> qS`, `phi_S -> phiS`, `theta_K -> qK`,
  `phi_K -> phiK`), and the dict is ordered exactly as `GenerateEMRIWaveform` expects positionally.
  `chi2` is *not* in the dict: FEW takes it as a trailing positional extra, which is what
  `StableEMRIFisher`'s `add_param_args` produces.

`fisher_params` is the set of parameters the corresponding PE run actually samples: the full 1PA
vector minus the YAML's `fixed_params`, minus `x_I0` (always held at 1.0).

In [ ]:
# Injection parameters for the three review test cases, in stableemrifisher naming and
# in FEW's positional order.  Masses are detector-frame, i.e. YAML value * (1 + z).

REVIEW_CASES = {
    1: dict(
        config="Review_test_1_inj_1PA_rec_1PA.yaml",
        z=0.004999974556005059,
        params={
            "m1": 5000000.0,               # 4975124.504066709 * (1 + z)
            "m2": 100.0,                   #      99.50249008133419 * (1 + z)
            "a": 8.2e-05,
            "p0": 8.8123,
            "e0": 0.0,
            "xI0": 1.0,
            "dist": 0.022213015032605164,
            "qS": 2.4314546363447995,
            "phiS": 2.757554564287996,
            "qK": 2.6973649175810768,
            "phiK": 4.381692553882582,
            "Phi_phi0": 0.5917337285168199,
            "Phi_theta0": 6.13001602516006,
            "Phi_r0": 4.782381792256834,
        },
        chi2=0.5,
        fixed=["x_I0", "e0", "Phi_theta0", "Phi_r0"],
    ),
    2: dict(
        config="Review_test_2_inj_1PA_rec_1PA.yaml",
        z=0.1,
        params={
            "m1": 549945.0549945006,       # 499950.0499950005 * (1 + z)
            "m2": 441.0890010998901,       #    400.9900009999 * (1 + z)
            "a": 0.0,
            "p0": 29.388286969253294,
            "e0": 0.0,
            "xI0": 1.0,
            "dist": 3.9425974141549913,
            "qS": 2.4314546363447995,
            "phiS": 2.757554564287996,
            "qK": 2.6973649175810768,
            "phiK": 4.381692553882582,
            "Phi_phi0": 0.5917337285168199,
            "Phi_theta0": 6.13001602516006,
            "Phi_r0": 4.782381792256834,
        },
        chi2=0.9,
        fixed=["x_I0", "e0", "a", "Phi_theta0", "Phi_r0"],
    ),
    3: dict(
        config="Review_test_3_inj_1PA_rec_1PA.yaml",
        z=0.01,
        params={
            "m1": 5044955.666098902,       # 4995005.609998913 * (1 + z)
            "m2": 504.49556660989015,      #  499.5005609998912 * (1 + z)
            "a": 0.0,
            "p0": 11.288286969253296,
            "e0": 0.0,
            "xI0": 1.0,
            "dist": 0.4429032864588028,
            "qS": 1.89,
            "phiS": 0.67,
            "qK": 1.7753,
            "phiK": 3.82,
            "Phi_phi0": 0.5917337285168199,
            "Phi_theta0": 6.13001602516006,
            "Phi_r0": 4.782381792256834,
        },
        chi2=0.998,
        fixed=["x_I0", "e0", "Phi_theta0", "Phi_r0"],
    ),
}

# Waveform / response settings, identical across the three configs.
T_OBS = 2.0          # years
DT = 5.0             # s
EVOLVE_CHI1 = True
INCLUDE_1PA_AMPS = True
INSPIRAL_KWARGS = {"DENSE_STEPPING": 0, "max_init_len": 1000}
SUMMATION_KWARGS = {"pad_output": True}
AMPLITUDE_KWARGS = {}
RESPONSE_ORDER = 40  # "order" in the Response block

# YAML name -> stableemrifisher name, in FEW's positional order for the 1PA model.
SEF_NAME = {
    "M": "m1", "mu": "m2", "a": "a", "p0": "p0", "e0": "e0", "chi2": "chi2",
    "x_I0": "xI0", "d_L": "dist", "theta_S": "qS", "phi_S": "phiS",
    "theta_K": "qK", "phi_K": "phiK",
    "Phi_phi0": "Phi_phi0", "Phi_theta0": "Phi_theta0", "Phi_r0": "Phi_r0",
}
PARAM_NAMES_1PA = list(SEF_NAME)


def fisher_params(case):
    """Parameters the matching PE run samples: full vector minus fixed minus x_I0."""
    fixed = set(case["fixed"]) | {"x_I0"}
    return [SEF_NAME[n] for n in PARAM_NAMES_1PA if n not in fixed]


for i, c in REVIEW_CASES.items():
    print(f"case {i}: {len(fisher_params(c)):2d} params  {fisher_params(c)}")

#### Cross-check against the YAML files

Guards against the hard-coded dicts above drifting away from the configs.

In [ ]:
def check_against_yaml(case):
    cfg = yaml.safe_load((CONFIG_DIR / case["config"]).read_text())
    emri = cfg["Injection"]["EMRI"]
    wave = cfg["Injection"]["Waveform"]
    z = float(emri["z"])

    assert z == case["z"], f"z: {z} != {case['z']}"
    assert float(emri["chi2"]) == case["chi2"]
    assert sorted(cfg["Sampler"]["fixed_params"]) == sorted(case["fixed"])

    for yaml_name, sef_name in SEF_NAME.items():
        if yaml_name == "chi2":
            continue
        expected = float(emri[yaml_name])
        if yaml_name in ("M", "mu"):
            expected *= 1.0 + z
        got = case["params"][sef_name]
        assert np.isclose(got, expected, rtol=0, atol=0), \
            f"{yaml_name}/{sef_name}: {got!r} != {expected!r}"

    assert float(wave["T"]) == T_OBS and float(wave["dt"]) == DT
    assert bool(wave["evolve_chi1"]) is EVOLVE_CHI1
    assert bool(wave["include_1PA_amps"]) is INCLUDE_1PA_AMPS
    assert int(cfg["Response"]["order"]) == RESPONSE_ORDER
    assert cfg["Response"]["tdi_gen"] == "2nd generation"
    return True


for i, c in REVIEW_CASES.items():
    check_against_yaml(c)
    print(f"case {i}: matches {c['config']}")

### Waveform, response and Fisher setup

Deviations from the PE runs, all forced by what is available locally:

| | PE run | here |
|---|---|---|
| orbits | ESA trailing orbit file on the cluster | `lisatools` bundled `ESAOrbits` |
| TDI channels | `XYZ` with the full noise covariance | `AE` with diagonal SciRD TDI2 PSDs |
| noise | Mojito noise realisation | analytic `scirdv1`, no confusion foreground |
| response epoch | tied to the Mojito L1 `t0` | `t0` = 10000 s garbage buffer only |

`AE` versus `XYZ` is not an approximation as long as the `T` channel carries no signal; dropping
`T` is. The orbit and epoch differences move the antenna pattern slightly, so expect O(10%) level
differences in the sky-angle errors rather than exact agreement.

`deriv_type="direct"` is used by default. `"stable"` reaches into FEW internals
(`_amplitudes_from_trajectory` and friends) that are written against the Kerr flux models; it may
well work for `Circ1PAT1R` — it does handle trailing extra parameters such as `chi2` — but it is
not verified here, so it is left as an opt-in.

In [ ]:
def waveform_generator(case, T=T_OBS, dt=DT):
    """Bare FEW generator for the 1PAT1R model with this case's toggles."""
    return GenerateEMRIWaveform(
        Circ1PAT1R,
        return_list=False,
        frame="detector",
        inspiral_kwargs={**INSPIRAL_KWARGS, "evolve_primary": EVOLVE_CHI1},
        amplitude_kwargs={**AMPLITUDE_KWARGS, "zero_PA_amps_only": not INCLUDE_1PA_AMPS},
        sum_kwargs=dict(SUMMATION_KWARGS),
    )


def plunge_trimmed_T(case, T=T_OBS, dt=DT, trim_hours=6.0, _gen_cache={}):
    """
    Observation time in years, shortened if the secondary plunges inside T.

    StableEMRIFisher has its own `plunge_check`, but it only rewrites the waveform kwargs --
    `ResponseWrapper.__call__` overwrites `kwargs["T"]` with its own `Tobs`, so with a response
    attached the trimming is silently discarded.  Doing it here instead means the ResponseWrapper
    is *built* with the trimmed duration and the trimming actually takes effect.
    """
    key = case["config"]
    gen = _gen_cache.get(key) or _gen_cache.setdefault(key, waveform_generator(case, T, dt))
    traj = gen.waveform_generator.inspiral_generator
    p = case["params"]
    t_traj = traj(
        p["m1"], p["m2"], p["a"], p["p0"], p["e0"], p["xI0"], case["chi2"],
        Phi_phi0=p["Phi_phi0"], Phi_theta0=p["Phi_theta0"], Phi_r0=p["Phi_r0"],
        T=T, dt=dt,
    )[0]
    if t_traj[-1] < T * YRSID_SI - 1.0:
        t_end = t_traj[-1] - trim_hours * 3600.0
        print(f"  plunges at {t_traj[-1] / YRSID_SI:.4f} yr -> "
              f"using T = {t_end / YRSID_SI:.4f} yr (last {trim_hours:g} h dropped)")
        return t_end / YRSID_SI
    print(f"  no plunge within {T:g} yr")
    return T


def response_kwargs(T, dt=DT, tdi_chan="AE", orbits=None, t0=10_000.0):
    """ResponseWrapper kwargs for fastlisaresponse 1.1.17 (force_backend, orbits instance)."""
    if orbits is None:
        orbits = ESAOrbits(force_backend=FORCE_BACKEND)
    return dict(
        Tobs=T,
        dt=dt,
        index_lambda=8,        # phiS in the 1PA vector (chi2 is appended after Phi_r0)
        index_beta=7,          # qS
        t0=t0,
        flip_hx=True,          # FEW returns h+ - i hx
        is_ecliptic_latitude=False,
        remove_garbage="zero",
        force_backend=FORCE_BACKEND,
        orbits=orbits,
        order=RESPONSE_ORDER,
        tdi="2nd generation",
        tdi_chan=tdi_chan,
    )


def build_sef(case, T=None, dt=DT, tdi_chan="AE", orbits=None, t0=10_000.0,
              deriv_type="direct", der_order=4, Ndelta=8, filename=None):
    """StableEMRIFisher configured for the 1PAT1R waveform plus the LISA response."""
    if T is None:
        T = plunge_trimmed_T(case, dt=dt)

    return StableEMRIFisher(
        waveform_class=Circ1PAT1R,
        waveform_class_kwargs=dict(
            inspiral_kwargs={**INSPIRAL_KWARGS, "evolve_primary": EVOLVE_CHI1},
            amplitude_kwargs={**AMPLITUDE_KWARGS, "zero_PA_amps_only": not INCLUDE_1PA_AMPS},
            sum_kwargs=dict(SUMMATION_KWARGS),
        ),
        waveform_generator=GenerateEMRIWaveform,
        waveform_generator_kwargs={"return_list": False, "frame": "detector"},
        ResponseWrapper=ResponseWrapper,
        ResponseWrapper_kwargs=response_kwargs(T, dt, tdi_chan, orbits, t0),
        use_gpu=USE_GPU,
        deriv_type=deriv_type,
        der_order=der_order,
        Ndelta=Ndelta,
        T=T,
        dt=dt,
        plunge_check=False,     # already handled by plunge_trimmed_T
        filename=filename,
    ), T

#### Finite-difference step ranges

`Fisher_Stability` falls back to `geomspace(1e-4 * value, 1e-9 * value)` for the intrinsic
parameters, which misbehaves for the two parameters that sit near a boundary here:

* `a` is `8.2e-5` (case 1) or exactly `0.0` (cases 2, 3), so a value-scaled grid is either far too
  small or undefined. A fixed absolute grid is used instead.
* `chi2` would get steps up to `0.1 * chi2`, pushing `chi2 = 0.998` (case 3) above 1. The grid is
  capped at `1e-2` and `chi2` is registered in `sef.minmax` so that values near the edge switch to
  one-sided differences automatically.

In [ ]:
def delta_ranges(Ndelta=8):
    return dict(
        a=np.geomspace(1e-4, 1e-9, Ndelta),        # absolute: a is ~0 in all three cases
        chi2=np.geomspace(1e-2, 1e-7, Ndelta),     # absolute: keeps chi2 = 0.998 below 1
    )


# Bounds used by Fisher_Stability to pick central/forward/backward differences.
# a is already there ([0.05, 0.95]); chi2 lives on [-1, 1] and needs the same treatment.
CHI2_MINMAX = [-0.95, 0.95]

### SNR check

Cheap sanity pass before committing to the derivatives: build the response once per case and read
off the optimal SNR. Compare against the SNRs quoted for the PE runs.

Reference values from this setup (`AE`, TDI2, `ESAOrbits`, `scirdv1` without confusion foreground):

| case | plunges at | SNR |
|---|---|---|
| 1 | 1.519 yr | 975 |
| 2 | 1.499 yr | 199 |
| 3 | 1.435 yr | 160 |

All three plunge well inside the configured `T = 2 yr`, which is why `plunge_trimmed_T` matters here.

In [ ]:
def snr_only(case, T=None, dt=DT, tdi_chan="AE"):
    sef, T_used = build_sef(case, T=T, dt=dt, tdi_chan=tdi_chan)
    params = dict(case["params"])
    rho = sef.SNRcalc_SEF(
        *(list(params.values()) + [case["chi2"]]),
        use_gpu=USE_GPU,
        dt=dt,
        T=T_used,
    )
    return rho, T_used


snrs = {}
for i, c in REVIEW_CASES.items():
    print(f"--- case {i} ---")
    rho, T_used = snr_only(c)
    snrs[i] = rho
    print(f"  SNR (AE, TDI2, T = {T_used:.4f} yr) = {rho:.1f}\n")

### Fisher matrices

Measured on this machine (CPU, no CuPy): a bare `T = 2 yr`, `dt = 5 s` `Circ1PAT1R` waveform takes
about 1 s, but one **response** evaluation takes about **20 s**, and that is what dominates.

Per case, the number of response calls is roughly `n_params * Ndelta * der_order` for the stable-delta
search plus `n_params * der_order` for the matrix itself:

| settings | calls (11 params) | wall time |
|---|---|---|
| `der_order=2, Ndelta=3` | ~90 | ~30 min |
| `der_order=2, Ndelta=4` | ~110 | ~40 min |
| `der_order=4, Ndelta=8` | ~400 | ~2 h |

`live_dangerously=True` skips the stability search entirely (~`n_params * der_order` calls, a couple
of minutes) and falls back to a mass-ratio/SNR heuristic for the step sizes. Good for a first look,
not for numbers you would quote.

In [ ]:
def run_case(case_id, der_order=4, Ndelta=8, tdi_chan="AE", T=None,
             deriv_type="direct", live_dangerously=False, save=True):
    case = REVIEW_CASES[case_id]
    names = fisher_params(case)
    tag = f"review_test_{case_id}"

    print(f"=== case {case_id} ===")
    sef, T_used = build_sef(
        case, T=T, tdi_chan=tdi_chan, deriv_type=deriv_type,
        der_order=der_order, Ndelta=Ndelta,
        filename=str(OUT_DIR / tag) if save else None,
    )
    sef.minmax["chi2"] = CHI2_MINMAX

    fisher = sef(
        dict(case["params"]),                 # copied: sef mutates it to append chi2
        add_param_args={"chi2": case["chi2"]},
        param_names=names,
        delta_range=delta_ranges(Ndelta),
        live_dangerously=live_dangerously,
        plunge_check=False,
    )

    cov = np.linalg.inv(fisher)
    sigma = np.sqrt(np.diag(cov))
    result = {
        "fisher": fisher, "cov": cov, "sigma": dict(zip(names, sigma)),
        "names": names, "snr": float(np.sqrt(sef.SNR2)), "T": T_used,
        "deltas": dict(sef.deltas) if sef.deltas else None,
    }
    if save:
        np.savez(OUT_DIR / f"{tag}_fisher.npz", fisher=fisher, cov=cov,
                 names=np.array(names), snr=result["snr"], T=T_used)
    return result


def report(result, case_id):
    case = REVIEW_CASES[case_id]
    truths = {**case["params"], "chi2": case["chi2"]}
    print(f"\ncase {case_id}:  SNR = {result['snr']:.1f},  T = {result['T']:.4f} yr")
    print(f"{'param':>10} {'truth':>18} {'sigma':>14} {'sigma/truth':>14}")
    for n in result["names"]:
        s, t = result["sigma"][n], truths[n]
        rel = f"{s / abs(t):.3e}" if t != 0 else "--"
        print(f"{n:>10} {t:18.8g} {s:14.6e} {rel:>14}")

In [ ]:
# Quick pass over all three cases (~40 min each).  For production numbers use
#     results[case_id] = run_case(case_id, der_order=4, Ndelta=8)
# and for a first look in a couple of minutes
#     results[case_id] = run_case(case_id, der_order=2, live_dangerously=True)
results = {}
for case_id in (1, 2, 3):
    results[case_id] = run_case(case_id, der_order=2, Ndelta=4)
    report(results[case_id], case_id)

### Summary

Compare `sigma` here against the marginal posterior widths from the matching PE runs in
`validation/PE_test_runs/sampling_data/`.

In [ ]:
def summary_table(results):
    all_names = []
    for r in results.values():
        for n in r["names"]:
            if n not in all_names:
                all_names.append(n)
    header = f"{'param':>10} " + " ".join(f"{'case ' + str(i):>14}" for i in results)
    print(header)
    print("-" * len(header))
    for n in all_names:
        row = f"{n:>10} "
        for r in results.values():
            row += f"{r['sigma'][n]:14.4e} " if n in r["sigma"] else f"{'--':>14} "
        print(row)
    print("-" * len(header))
    print(f"{'SNR':>10} " + " ".join(f"{r['snr']:14.1f}" for r in results.values()))


summary_table(results)

In [ ]:
def plot_sigmas(results):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    all_names = []
    for r in results.values():
        for n in r["names"]:
            if n not in all_names:
                all_names.append(n)
    x = np.arange(len(all_names))
    width = 0.8 / len(results)
    for k, (case_id, r) in enumerate(results.items()):
        truths = {**REVIEW_CASES[case_id]["params"], "chi2": REVIEW_CASES[case_id]["chi2"]}
        vals = [
            r["sigma"][n] / abs(truths[n]) if n in r["sigma"] and truths[n] != 0 else np.nan
            for n in all_names
        ]
        ax.bar(x + k * width, vals, width, label=f"case {case_id} (SNR {r['snr']:.0f})")
    ax.set_xticks(x + 0.4 - width / 2)
    ax.set_xticklabels(all_names, rotation=45, ha="right")
    ax.set_yscale("log")
    ax.set_ylabel(r"$\sigma_\theta / |\theta|$")
    ax.set_title("Fisher fractional errors, review test cases (1PAT1R, AE, TDI2)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    return fig


plot_sigmas(results);